# Part 5 — Grant the missing reader role and deploy registered model v3 only

ใช้ WSL / kernel `itcs355-lab3` เดิม รัน code cell เดียว

ผล audit 16:24 ยืนยัน:
- serving identity ถูกต้อง และ AcrPull มีแล้ว
- ขาด Storage Blob Data Reader บน Storage account `itcs3556688012`
- model `itcs355-6688012:3` ลงทะเบียนแล้ว validation PR-AUC 0.4024465128 ผ่าน gate

ผลกระทบเมื่อรัน cell นี้:
1. อ่านตรวจ state/model version/identity/สิทธิ์/compute และ parse job ก่อนแก้ cloud
2. เพิ่มเฉพาะ Storage Blob Data Reader ให้ `itcs355-6688012-lab5-serve` บน Storage account เดิมหากยังไม่มี
   สิทธิ์นี้อ่าน Blob ใน account นี้ได้ ไม่ใช่สิทธิ์เขียนหรือสิทธิ์ระดับ subscription
3. เผื่อเวลา role propagation 90 วินาที พร้อม progress แล้วรัน job ชื่อคงที่ `...-deploy-repair`
4. job มีเฉพาะ deploy_staging ใช้ output จาก register job เดิม และ frozen source เดิม
5. ตรวจ staging กับ prediction ว่าตอบเป็น model v3; หาก fail อ่าน job/container logs ให้อัตโนมัติ

**ไม่มี training, registration, compute/identity setup, Git commit หรือ push**
มีค่า compute สำหรับ release job และ staging DS2_v2 ตามการใช้งานจริง
งาน pipeline เดิมยังเป็น Failed; จะแยกหลักฐาน deployment continuation ตามจริง
รัน cell ซ้ำจะติดตาม job ชื่อเดิม ไม่ส่ง attempt ใหม่
หากสำเร็จ staging ยังเปิดอยู่ ต้องเก็บหลักฐานและ cleanup ต่อ ส่ง SEND THIS SUMMARY กลับมาครับ


In [ ]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/scripts/repair_deployment.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
print("Deployment-only repair — one scoped reader grant and existing model v3; no training.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.repair_deployment"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise
if exit_code:
    raise RuntimeError("Deployment repair หยุด: ส่ง summary/error ด้านบน ไม่ต้องเริ่ม training ใหม่ครับ")
print("ส่ง SEND THIS SUMMARY กลับมาครับ staging อาจยังคิดเงิน ต้องเก็บหลักฐานและ cleanup ต่อ", flush=True)
